# Re100 FEM: mesh and time-step refinement

This study assesses the numerical reliability of the FEM reference data used in our analysis by testing sensitivity to mesh resolution and time-step size.

Compare the original settings with two finer meshes and two smaller time steps. The acceptance threshold is **0.5%** for each of the four quantities below.

### Force coefficients

$$
C_D(t)=\frac{F_x(t)}{\frac{1}{2}\rho U^2D},\qquad
C_L(t)=\frac{F_y(t)}{\frac{1}{2}\rho U^2D}.
$$

$F_x$ and $F_y$ are the fluid forces on the cylinder **per unit span**, including pressure and viscous forces. $x$ is the flow direction; $y$ is the transverse direction. $\rho$ is fluid density, $U$ is inlet speed, and $D$ is cylinder diameter. **Drag $C_D$ measures streamwise resistance; lift $C_L$ measures transverse loading.** Here $\rho=U=D=1$, so $C_D=2F_x$ and $C_L=2F_y$.

### Quantities

For a complete shedding cycle from $t_a$ to $t_b$, let $T=t_b-t_a$.

**Mean drag coefficient**, $\overline{C_D}$: the average streamwise load.

$$
\overline{C_D}=\frac{1}{T}\int_{t_a}^{t_b}C_D(t)\,dt.
$$

**Drag fluctuation RMS**, $C'_{D,\mathrm{rms}}$: the size of drag oscillations about the cycle mean.

$$
C'_{D,\mathrm{rms}}=\sqrt{\frac{1}{T}\int_{t_a}^{t_b}\left(C_D(t)-\overline{C_D}\right)^2\,dt}.
$$

**Lift RMS**, $C_{L,\mathrm{rms}}$: the size of transverse loading. With near-zero mean lift, it measures lift oscillations.

$$
C_{L,\mathrm{rms}}=\sqrt{\frac{1}{T}\int_{t_a}^{t_b}C_L(t)^2\,dt}.
$$

**Strouhal number**, $St$: the dimensionless shedding frequency.

$$
St=\frac{fD}{U},\qquad f=\frac{1}{T}.
$$

**Mean drag and drag fluctuation RMS test different features:** the steady load and its oscillation size. Similar mean drag can conceal different oscillations; subtracting the mean prevents it from dominating the RMS.

Here $U=D=1$, so $St=1/T$. Each reported quantity $q$ is the average over the **last three complete cycles within $60\le t\le90$**. All comparisons use data at $t\le90$.

### Relative error

For successive settings, $i+1$ is the finer mesh or smaller time step:

$$
\varepsilon_i(q)=100\frac{|q_{i+1}-q_i|}{|q_{i+1}|}\quad[\%].
$$

Also compare the original setting directly with the finest tested setting:

$$
\varepsilon_{\mathrm{ref}}(q)=100\frac{|q_{\mathrm{finest}}-q_{\mathrm{original}}|}{|q_{\mathrm{finest}}|}\quad[\%].
$$

All four quantities must meet the **0.5%** threshold. These are differences in cycle statistics over the selected late-time interval.

This notebook reads saved force histories and writes regenerated plots, statistics and assessments to the selected `Source_Data/Generated_Outputs/CFD_Refinement_Analysis` directory. It does not modify the archived `Source_Data` files.

In [ ]:
from pathlib import Path
import json
import os
import hashlib
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Locate the code root from the current directory or its parents.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'Source/CFD/refinement.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook inside the repository containing Source/CFD/refinement.py')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from Source.data_locations import source_file, source_data_root
from Source.CFD.refinement import generated_data_root
from Source.CFD.inputs import steady_input
from Source.CFD.refinement_analysis import late_window_statistics, relative_changes

# Relative-error limit: 0.5%.
RELATIVE_ERROR_LIMIT = 0.005
COMPARISON_START = 60.0
COMPARISON_END = 90.0

DATA_ROOT = source_data_root()
# The solver-output directory contains the archived inputs for this analysis.
RESULTS = Path(os.environ.get('KVN_CFD_REFINEMENT_OUTPUT', generated_data_root())).expanduser()
if not RESULTS.is_absolute():
    RESULTS = ROOT / RESULTS
RESULTS = RESULTS.resolve()
if not RESULTS.is_relative_to(DATA_ROOT):
    raise ValueError('The stored refinement results must be inside Source_Data')
# Write fresh analyses in the selected writable dataset, outside Codes.
OUTPUT = DATA_ROOT / 'Generated_Outputs/CFD_Refinement_Analysis'
FIGURES = OUTPUT / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
from Figure_Plots.diagnostic_style import diagnostic_style, COLORS, TIME_LABEL
print('Stored results:', (Path('Source_Data') / RESULTS.relative_to(DATA_ROOT)).as_posix())
print('Analysis outputs:', OUTPUT.as_posix())
print('Re-run all cells to read newly completed runs. No CFD is launched by this notebook.')

## Original settings and added tests

### Original reference

| Parameter | Setting |
|---|---|
| Reynolds number | $Re=UD/\nu=100$ |
| Scales | $U=D=\rho=1$; $\nu=0.01$ |
| Domain | $[-10,20]\times[-10,10]$; cylinder centred at $(0,0)$, diameter $D=1$ |
| FEM mesh | 9,314 triangles over the full domain; spacing factor 1.0 |
| Degrees of freedom | 37,640 velocity; 4,753 pressure |
| Elements / quadrature | Taylor–Hood P2/P1; quadrature order 5 |
| Time step / saved duration | $\Delta t=0.02$; $0\le t\le90$ |
| Time integration | IMEX Euler first step, then BDF2/AB2; steady convection Jacobian implicit |
| Inlet, top and bottom | $\mathbf{u}=(1,0)$ |
| Cylinder | $\mathbf{u}=0$ |
| Outlet | Natural condition $\nu\partial_n\mathbf{u}-p\mathbf{n}=0$ |
| Initial state | Symmetric steady flow plus the `train` perturbation: Gaussian streamfunction amplitude 0.02, centre $(2,0)$, width 0.7; cylinder cutoff and discrete divergence-free projection |

Time is scaled by $D/U$. The 128×64 output sampling grid is separate from the FEM mesh.

### Added calculations

| Run | Mesh spacing factor | Full-domain triangles | $\Delta t$ | Final $t$ |
|---|---:|---:|---:|---:|
| Original reference | 1.00 | 9,314 | 0.020 | 90 |
| Time refinement 1 | 1.00 | 9,314 | 0.010 | 90 |
| Time refinement 2 | 1.00 | 9,314 | 0.005 | 90 |
| Mesh refinement 1 | 0.80 | 15,902 | 0.020 | 90 |
| Mesh refinement 2 | 0.64 | 25,960 | 0.020 | 90 |

Mesh tests hold $\Delta t=0.02$ fixed; time-step tests hold the original mesh fixed. All runs use the same domain, boundary conditions, elements, scheme and perturbation parameters. Each refined mesh gets its own steady solution. Smaller spacing factors refine both the fluid mesh and the polygonal cylinder boundary.

The original reference is read directly from `Source_Data`. The four additional force histories are stored in `Source_Data/Generated/CFD_Refinement`; only their $0\le t\le90$ samples are used. The plots compare $60\le t\le90$, and the statistics use the last three complete cycles in that interval. Future refinement runs also end at $t=90$.


In [ ]:
from matplotlib.tri import Triangulation
with np.load(steady_input()) as data:
    points = data['mesh_points']
    triangles = data['mesh_triangles']
# Display the upper half actually stored in the symmetric steady input.
tri = Triangulation(points[0], points[1], triangles.T)
with diagnostic_style():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax in axes:
        ax.triplot(tri, linewidth=.3, color=COLORS[0])
        ax.set_aspect('equal')
        ax.set(xlabel='x / D', ylabel='y / D')
    axes[0].set(title=f'Reference upper-half mesh: {triangles.shape[1]:,} triangles',
                xlim=(-10, 20), ylim=(0, 10))
    axes[1].set(title='Cylinder and near wake', xlim=(-1, 4), ylim=(0, 2))
    fig.tight_layout()
    fig.savefig(FIGURES / 'reference_mesh.png', bbox_inches='tight')
    plt.show()


## Original force history

The plots show the original $0<t\le90$ history and a $60\le t\le90$ zoom. The $t=0$ force is excluded because the perturbed initial velocity has no solved pressure.

Cycle boundaries are upward zero crossings of $C_L$, located by cubic interpolation. Statistics use complete cycles without smoothing and average the last three complete cycles within $60\le t\le90$.


In [ ]:
archive_path = source_file('Cylinder_Forces_Re100_20261110/archived_replay/forces.csv')
archived = pd.read_csv(archive_path).dropna(subset=['C_D', 'C_L'])
with diagnostic_style():
    fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex='col')
    for row, key in enumerate(('C_D', 'C_L')):
        for col, start in enumerate((0., 60.)):
            selected = archived[archived.time >= start]
            axes[row, col].plot(selected.time, selected[key], lw=1.2)
            axes[row, col].set(ylabel={'C_L': r'$C_L$', 'C_D': r'$C_D$'}[key], xlabel=TIME_LABEL)
    axes[0, 0].set_title('Original force history')
    axes[0, 1].set_title('Original force history: 60 ≤ t ≤ 90')
    fig.tight_layout()
    fig.savefig(FIGURES / 'published_forces.png', bbox_inches='tight')
    plt.show()


## Mesh refinement

Hold $\Delta t=0.02$ fixed and compare **mesh spacing factors 1.00, 0.80 and 0.64**. The plots show $C_L(t)$ and $C_D(t)$ over $60\le t\le90$, using the same time axis without phase alignment.

In [ ]:
# Load completed results. Internal directory names are used only to read files.
if not (RESULTS / 'report.json').is_file():
    raise FileNotFoundError('Generate the refinement data first: python -m Source.CFD.refinement')
report = json.loads((RESULTS / 'report.json').read_text())
if report['state'] != 'completed' or len(report['runs']) != 5:
    raise RuntimeError('The original reference and four refinement histories are required.')
expected_settings = {(1., .02), (1., .01), (1., .005), (.8, .02), (.64, .02)}
if {(float(run['factor']), float(run['dt'])) for run in report['runs']} != expected_settings:
    raise RuntimeError('The report must contain the five declared mesh/time-step settings.')
missing = [run['force_file'] for run in report['runs']
           if not (DATA_ROOT / run['force_file']).is_file()]
if missing:
    raise FileNotFoundError('Missing refinement force histories: ' + ', '.join(missing))
# Recompute all four quantities from the saved forces; retain the original run report.
runs = []
for stored_run in report['runs']:
    history = pd.read_csv(DATA_ROOT / stored_run['force_file'])
    if not np.isclose(history.time.max(), COMPARISON_END, rtol=0., atol=1e-9):
        raise ValueError('Each force history must end at nondimensional time 90.')
    statistics = late_window_statistics(history, start=COMPARISON_START, end=COMPARISON_END)
    runs.append({**stored_run, 'metrics': statistics})

# The stored field "factor" is the mesh spacing factor defined in the setup.
mesh_runs = sorted([run for run in runs if run['dt'] == .02],
                   key=lambda run: run['factor'], reverse=True)
time_step_runs = sorted([run for run in runs if run['factor'] == 1.],
                        key=lambda run: run['dt'], reverse=True)


def plot_force_comparison(selected_runs, labels, filename):
    """Plot drag and lift on the same nondimensional-time interval for every setting."""
    with diagnostic_style():
        fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
        colors = COLORS
        line_styles = ('-', '--', ':')
        for run, label, color, style in zip(selected_runs, labels, colors, line_styles):
            history = pd.read_csv(DATA_ROOT / run['force_file'])
            history = history[(history.time >= COMPARISON_START) & (history.time <= COMPARISON_END)]
            for ax, column, symbol in zip(axes, ('C_D', 'C_L'), (r'$C_D$', r'$C_L$')):
                ax.plot(history.time, history[column], color=color, linestyle=style,
                        linewidth=1.4, label=label)
                ax.set(xlabel=TIME_LABEL, ylabel=symbol, xlim=(COMPARISON_START, COMPARISON_END))
        handles, legend_labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, legend_labels, loc='upper center', ncol=3,
                   fontsize=9, frameon=False)
        fig.tight_layout(rect=(0, 0, 1, .90))
        fig.savefig(FIGURES / filename, bbox_inches='tight')
        plt.show()


plot_force_comparison(
    mesh_runs,
    [f"Mesh spacing factor = {run['factor']:.2f}" for run in mesh_runs],
    'mesh_refinement_forces.png',
)

## Time-step refinement

Hold **mesh spacing factor 1.00** fixed and compare **$\Delta t=0.020$, $0.010$ and $0.005$**. Use the same $60\le t\le90$ interval.

In [ ]:
plot_force_comparison(
    time_step_runs,
    [rf"$\Delta t = {run['dt']:.3f}$" for run in time_step_runs],
    'time_step_refinement_forces.png',
)

## Relative errors and conclusion

The table uses the four quantities and error definitions given at the start. For each test, the first two rows give **adjacent-setting errors** $\varepsilon_i$; the third gives the **original-to-finest error** $\varepsilon_{\mathrm{ref}}$. All entries are percentages; the threshold is **0.5%**.

In [ ]:
# Use the same quantity names and symbols as the definitions in the first cell.
quantity_columns = {
    'mean_CD': 'Mean drag coefficient',
    'rms_CD_fluct': 'Drag fluctuation RMS',
    'rms_CL': 'Lift RMS',
    'St': 'Strouhal number',
}
comparison_rows = []
for selected_runs, setting, setting_name, digits in (
    (mesh_runs, 'factor', 'Mesh spacing factor', 2),
    (time_step_runs, 'dt', 'Δt', 3),
):
    # First two pairs are adjacent refinements; the last compares original to finest.
    for first, second, comparison in ((0, 1, 'Adjacent'), (1, 2, 'Adjacent'),
                                       (0, 2, 'Original to finest')):
        previous, current = selected_runs[first], selected_runs[second]
        difference = relative_changes(previous['metrics'], current['metrics'])
        comparison_rows.append({
            'Setting': setting_name,
            'Comparison': f"{previous[setting]:.{digits}f} → {current[setting]:.{digits}f}",
            'Error': comparison,
            **{label: 100*difference[key] for key, label in quantity_columns.items()},
        })

error_table = pd.DataFrame(comparison_rows)
display(error_table.style.hide(axis='index').format(
    {label: '{:.4f}%' for label in quantity_columns.values()}
))
error_table.to_csv(OUTPUT / 'refinement_errors.csv', index=False)

verdicts = {}
for selected_runs, label in ((mesh_runs, 'Mesh'), (time_step_runs, 'Time step')):
    first_pair = relative_changes(selected_runs[0]['metrics'], selected_runs[1]['metrics'])
    last_pair = relative_changes(selected_runs[1]['metrics'], selected_runs[2]['metrics'])
    original_to_finest = relative_changes(selected_runs[0]['metrics'], selected_runs[2]['metrics'])
    passed = max(*first_pair.values(), *last_pair.values(), *original_to_finest.values()) <= RELATIVE_ERROR_LIMIT
    conclusion = 'meets' if passed else 'does not meet'
    conclusion += f' the {100*RELATIVE_ERROR_LIMIT:g}% criterion'
    verdicts[label] = dict(passed=passed, first_pair_relative_change=first_pair,
                          last_pair_relative_change=last_pair,
                          original_to_finest_relative_change=original_to_finest)
    print(f"{label}: the original setting {conclusion} over 60 ≤ t ≤ 90; "
          f"largest original-to-finest error = {100*max(original_to_finest.values()):.3f}%.")

# Save the revised assessment separately from the original calculation provenance.
assessment = {
    'relative_error_limit': RELATIVE_ERROR_LIMIT,
    'comparison_window': [COMPARISON_START, COMPARISON_END],
    'quantities': quantity_columns,
    'runs': runs,
    'verdicts': verdicts,
    'force_file_sha256': {
        run['name']: hashlib.sha256((DATA_ROOT / run['force_file']).read_bytes()).hexdigest()
        for run in runs
    },
    'analysis_source_sha256': {
        name: hashlib.sha256((ROOT / name).read_bytes()).hexdigest()
        for name in ('Source/CFD/refinement_analysis.py', 'Source/CFD/refinement_metrics.py')
    },
    'source_report_sha256': hashlib.sha256((RESULTS / 'report.json').read_bytes()).hexdigest(),
}
(OUTPUT / 'four_quantity_assessment.json').write_text(
    json.dumps(assessment, indent=2, allow_nan=False) + '\n'
)
pd.DataFrame([
    {'Mesh spacing factor': run['factor'], 'Δt': run['dt'],
     **{label: run['metrics'][key] for key, label in quantity_columns.items()}}
    for run in runs
]).to_csv(OUTPUT / 'four_quantity_statistics.csv', index=False)



## Scope and references

These tests measure mesh and time-step sensitivity separately in the original data’s late-time interval, 60 ≤ t ≤ 90. They do not estimate domain-size error, combined space–time error or a formal Grid Convergence Index.

- [NASA: Spatial convergence](https://www.grc.nasa.gov/www/wind/valid/tutorial/spatconv.html) — refinement and discretization-error assessment.
- [INGV GALES: Fixed cylinder at Re100](https://gales.pi.ingv.it/fixed-cylinder-re-100/) — periodic forces and Strouhal number. Its different boundary conditions preclude using its coefficients as acceptance targets.
